### ファイルインポート

In [8]:
import pandas as pd
import numpy as np


In [9]:
# --- ファイル読み込み ---
df_oof_all = pd.read_parquet("driver_oof.parquet", engine="pyarrow")
df_shap = {
    "1:Credit":pd.read_parquet("driver_shap_credit.parquet", engine="pyarrow"),
    "2:Bond":pd.read_parquet("driver_shap_bond.parquet", engine="pyarrow"),
    "3:Mix":pd.read_parquet("driver_shap_mix.parquet", engine="pyarrow")
}
df_oof_ev = pd.read_parquet("driver_oof_ev.parquet", engine="pyarrow")
df_daily = pd.read_parquet("driver_daily.parquet", engine="pyarrow")
df_driver = pd.read_parquet("driver_driver.parquet", engine="pyarrow")
df_features = pd.read_parquet("driver_features.parquet", engine="pyarrow")
df_label = pd.read_parquet("driver_label.parquet", engine="pyarrow")

### 結果表示関数

In [10]:
# 期待値のヒストグラム

def _chk_ev_hist(df_oof_ev):
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 1. 基本統計量の確認（平均、最小、最大、四分位数）
    print("=== risk_sum の基本統計量 ===")
    print(df_oof_ev['risk_sum'].describe())

    plt.figure(figsize=(10, 6))
    sns.histplot(df_oof_ev['risk_sum'], bins=50, kde=True, color='royalblue')
    plt.title('Distribution of Risk Sum (Credit + Bond Probability)')
    plt.xlabel('Risk Sum Value')
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.3)
    plt.axvline(x=0.5, color='red', linestyle='--', label='Threshold 0.5')
    plt.legend()
    plt.show()


In [11]:
# Accuracy
def _chk_accuracy(df_oof_all):
    df_prob = df_oof_all[["1:Credit", "2:Bond", "3:Mix"]]
    df_prob['dominant_regime'] = df_prob.idxmax(axis=1).str.split(':').str[0].astype(int)
    teacher = df_oof_all["actual_regime"]
    ai = df_prob["dominant_regime"]
    teacher, ai = teacher.align(ai, join="inner")

    from sklearn.metrics import classification_report,confusion_matrix
    print("\nAccuracyの結果レポート")
    print(classification_report(teacher, ai))
    print(confusion_matrix(teacher, ai))

In [12]:
# ev_rankの20にリターン
def _chk_ev(df_oof_ev):
    ev_summary = df_oof_ev.groupby('ev_rank', observed=True)['actual_return'].agg(['mean',"median", 'count'])
    print("\n期待値の結果レポート")
    print(ev_summary.reindex(['Safe', 'Neutral', 'Caution', 'High Risk', 'CRITICAL']))

### CRITICAL

In [13]:
# SHAP 特徴量分析
def _da_CRITICAL_shap(df_oof_all, df_oof_ev, df_shap, df_driver):
    # ------------ 1. 分類 -------------
    # 特徴量列を追加
    df_oof_ev = df_oof_ev.join(df_driver, how='left')

    # CRITICALのみ
    critical_df = df_oof_ev[df_oof_ev['ev_rank'] == 'CRITICAL'].copy()
    critical_df = critical_df.join(df_oof_all[["1:Credit","2:Bond","3:Mix"]])
    #print(critical_df)

    # 反発と下落に分ける
    rebound_df = critical_df[critical_df['actual_return'] > 0]
    hit_df = critical_df[critical_df['actual_return'] <= 0]

    # 反発の中でも周辺のrisk_sumが低い期間を抽出 = AIのミス
    pd.set_option('display.max_rows', None)
    print(rebound_df)
    print(hit_df)
    rebound_df = rebound_df.loc["2023-10-01":"2023-10-31"]
    print("反発の中でも周辺のrisk_sumが低くマクロが壊れていない可能性が高い期間は、2023-10-01-2023-10-31")

    # ------------ 2. 犯人 -------------
    # 反発と下落でそれぞれCreditとBondの確率平均を出す
    for name, target in [("リバウンド", rebound_df), ("的中", hit_df)]:
        if not target.empty:
            p_credit = target['1:Credit'].mean()
            p_bond   = target['2:Bond'].mean()
            print(f"【{name}】の平均確率 -> Credit: {p_credit:.3f} / Bond: {p_bond:.3f}")

    # 2. どちらのレジュームが「ノイズ」を呼んでいるか？
    if rebound_df['2:Bond'].mean() > rebound_df['1:Credit'].mean():
        print("\n[診断] 反発は主に『Bond（金利急変）』由来で発生しています。")
    else:
        print("\n[診断] 反発は主に『Credit（スプレッド拡大）』由来で発生しています。")

    # ------------ 3. SHAP -------------
    rebound_indices = rebound_df.index
    rebound_shap = df_shap["1:Credit"].loc[rebound_indices]
    hit_indices = hit_df.index
    hit_shap = df_shap["1:Credit"].loc[hit_indices]

    print("\n=== 反発を誘発した特徴量 Top 5 (SHAP平均) ===")
    print(rebound_shap.mean().sort_values(ascending=False).head(5))
    print("\n=== 下落を誘発した特徴量 Top 5 (SHAP平均) ===")
    print(hit_shap.mean().sort_values(ascending=False).head(5))

    # ------------ 4. 特徴量 -------------
    features_to_analyze = [
        'VIX_z252', 'VVIX_z252', 'MOVE_z252', 'MOVE_vov', 'hy_z252',
        'SOFR_vol_spike', 'Term_Premium_z252', 'Credit_Equity_Divergence',
        'Term_Premium_diff5_z252', 'DFII10_diff5_zscore',
        'Stock_Bond_Corr_zscore', 'Equity_Gold_Ratio_zscore',
        'Flight_to_Safety_zscore', 'tlt_hy_ratio_z252'
        ]
    stats_rebound = critical_df.loc[rebound_indices, features_to_analyze].describe().T
    stats_hit    = critical_df.loc[hit_indices, features_to_analyze].describe().T

    analysis = pd.DataFrame({
        'mean_hit (Win)': stats_hit['mean'],
        'mean_rebound(Loss)': stats_rebound['mean'],
        'median_hit': stats_hit['50%'],
        'median_rebound': stats_rebound['50%']
    })

    # 差分を計算。この値が大きいほど、その指標は見分ける武器になる
    analysis['diff_mean'] = analysis['mean_hit (Win)'] - analysis['mean_rebound(Loss)']

    print(f"=== CRITICAL内部解剖レポート ===")
    print(f"下落(リターン正): {len(hit_indices)} 日")
    print(f"反発(リターン負): {len(rebound_indices)} 日")
    print("-" * 50)
    print(analysis.sort_values('diff_mean', key=abs, ascending=False))

In [14]:
# フィルター
def _da_CRITICAL_filter(df_oof_all, df_oof_ev, df_shap, df_driver):

    df_oof_ev = df_oof_ev.join(df_driver, how='left')
    mask_target = (df_oof_ev['ev_rank'] == 'CRITICAL') & (df_oof_ev['Credit_Equity_Divergence'] <= 1.0)
    mask_rebound = (df_oof_ev['ev_rank'] == 'CRITICAL') & (df_oof_ev['Credit_Equity_Divergence'] > 1.0)
    features = [
        'VIX_z252', 'VVIX_z252', 'MOVE_z252', 'MOVE_vov', 'hy_z252',
        'SOFR_vol_spike', 'Term_Premium_z252', 'Credit_Equity_Divergence',
        'Term_Premium_diff5_z252', 'DFII10_diff5_zscore',
        'Stock_Bond_Corr_zscore', 'Equity_Gold_Ratio_zscore',
        'Flight_to_Safety_zscore', 'tlt_hy_ratio_z252'
        ]
    stats_target = df_oof_ev.loc[mask_target, features].describe().T
    stats_rebound = df_oof_ev.loc[mask_rebound, features].describe().T
    
    comparison = stats_target[['mean', '50%']].join(
        stats_rebound[['mean', '50%']], 
        lsuffix='_Target(21d)', 
        rsuffix='_Rebound(37d)'
    )
    comparison['diff_mean'] = comparison['mean_Target(21d)'] - comparison['mean_Rebound(37d)']
    print("=== 的中(21日) vs リバウンド(37日) 特徴量比較レポート ===")
    print(comparison.sort_values('diff_mean', key=abs, ascending=False))

### バックテスト

In [15]:
# バックテスト
def calc_mdd(cum_series):
    dd = (cum_series / cum_series.cummax() - 1)
    mdd = dd.min()
    mdd_idx = dd.idxmin()
    return mdd, mdd_idx

def _cal_driver_signal(df_oof_ev,df_oof_all,df_daily,START_DATE,END_DATE):
    # 1. データ準備とラグの処理
    df_bt = df_oof_ev.join(df_oof_all, how='left').sort_index().copy()
    sp500_ret = df_daily["^GSPC"].pct_change().dropna().rename("sp500_ret")
    df_bt = df_bt.join(sp500_ret, how='left').sort_index().copy()
    df_bt = df_bt.loc[START_DATE:END_DATE]

    # 2-1. ロジックA Credit単独(>0.4) または Hybrid Risk Score(>0.6)
    credit_threshold = 0.4
    hybrid_threshold = 0.6
    df_bt['risk_score'] = df_bt['1:Credit'] + (df_bt['2:Bond'] * 0.5)
    df_bt['macro_danger'] = (
        (df_bt['ev_rank'].isin(['CRITICAL', 'High Risk'])) & 
        ((df_bt['1:Credit'] > credit_threshold) | (df_bt['risk_score'] > hybrid_threshold))
    )

    # 2-2. ロジックB - ロジックAにパラシュート追加（価格アクションによる緊急回避）
    # 直近5日のボラティリティに対して-2.0倍の急落を検知
    #df_bt['vol_5d'] = df_bt['sp500_ret'].rolling(5).std()
    #df_bt['vol_5d'] = df_bt['sp500_ret'].shift(1).rolling(5).std()
    vol_clean = sp500_ret.rolling(5).std().rename("vol_5d")
    df_bt = df_bt.join(vol_clean, how='left')
    df_bt['price_crash'] = df_bt['sp500_ret'] < (-2.0 * df_bt['vol_5d'])

    # 「マクロの地盤が緩んでいる（Caution以上）」かつ「価格が急落」した場合のみパラシュート
    #df_bt['parachute_active'] = (df_bt['risk_sum'] > hybrid_threshold) & df_bt['price_crash']
    df_bt['parachute_active'] = (df_bt['risk_sum'] > 0.5) & df_bt['price_crash']
    
    return df_bt

def _back_test(df_daily, df_oof_ev, df_oof_all,START_DATE,END_DATE):

    df_bt = _cal_driver_signal(df_oof_ev,df_oof_all,df_daily,START_DATE,END_DATE)

    # --- 戦略リターンの計算 (シグナルのラグ) ---
    results_return = []
    results_mdd = []
    results_mdd_idx =[]
    for lag in range(21):

        # 前日の終値で出た判定（ev_rank）を、今日のリターン（sp500_ret）に適用する
        #signal = df_bt['ev_rank'].shift(lag)
        basic_signal = df_bt['macro_danger'].shift(lag).fillna(False)
        parachute_signal = df_bt['parachute_active'].shift(1).fillna(False)
        df_bt['rtm_signal'] = basic_signal | parachute_signal

        # 昨日のシグナルがCRITICALなら、今日のリターンは0（現金退避）
        return_strategy = np.where(
            #signal.isin(danger_ranks),
            #basic_signal,
            #parachute_signal,
            df_bt['rtm_signal'],
            0.0,
            df_bt['sp500_ret']
        )
        return_strategy = pd.Series(return_strategy, index=df_bt.index)
        # 3. 累積リターンの計算
        cum_strategy  = (1 + return_strategy.fillna(0)).cumprod()
        mdd_strategy, mdd_idx = calc_mdd(cum_strategy)
        results_return.append(cum_strategy)
        results_mdd.append(mdd_strategy)
        results_mdd_idx.append(mdd_idx)

    df_strategy_return = pd.concat(results_return, axis=1)
    df_strategy_return.columns = [f"lag_{i}" for i in range(21)]
    df_strategy_mdd = pd.Series(results_mdd, index=[f"lag_{i}" for i in range(21)])
    df_strategy_mdd_idx = pd.Series(results_mdd_idx, index=[f"lag_{i}" for i in range(21)])

    df_bt['cum_benchmark'] = (1 + df_bt['sp500_ret'].fillna(0)).cumprod()
    mdd_bench, mdd_bench_idx = calc_mdd(df_bt['cum_benchmark'])

    print("=== 現実的バックテスト結果（lag sweep） ===")
    print(f"{START_DATE}～{END_DATE}")
    print(
        f"Buy & Hold 最終: {df_bt['cum_benchmark'].iloc[-1]:.2f}倍 "
        f"/ MDD: {mdd_bench:.1%} (at {mdd_bench_idx})"
    )
    print(f"パラシュート検知数: {df_bt['parachute_active'].sum()}")
    print(f"RTM Strategy  リターン:")
    final_returns = df_strategy_return.iloc[-1]

    summary = pd.DataFrame({
        "lag": range(21),
        "final_return": df_strategy_return.iloc[-1].values,
        "MDD": df_strategy_mdd.values,
        "MDD_date": df_strategy_mdd_idx
    }).set_index("lag")

    print(summary.sort_values("final_return", ascending=False).sort_index())

    # shift(1) で発生する NaN を False で埋めることでエラーを回避
    mask = df_bt['parachute_active'].shift(1).fillna(False)
    # 損失（sp500_retがマイナス）を回避した順にソート
    rescue_log = df_bt[mask].sort_index()

    # 回避した損失が特に大きい「ワースト10」を表示
    print("=== RTM パラシュート救済リスト（回避した損失のワースト10） ===")
    # 回避できたはずのマイナスリターン（sp500_ret）が深い順に表示
    print(rescue_log[['sp500_ret', '1:Credit', '2:Bond', 'risk_sum', 'risk_score']].sort_values('sp500_ret'))


In [16]:
START_DATE = "2025-03-01"
END_DATE = "2025-05-01"

In [17]:
_back_test(df_daily, df_oof_ev,df_oof_all,START_DATE,END_DATE)

=== 現実的バックテスト結果（lag sweep） ===
2025-03-01～2025-05-01
Buy & Hold 最終: 0.97倍 / MDD: -14.8% (at 2025-04-08 00:00:00)
パラシュート検知数: 2
RTM Strategy  リターン:
     final_return       MDD   MDD_date
lag                                   
0        0.933509 -0.111074 2025-04-16
1        1.049968 -0.033879 2025-03-28
2        0.959909 -0.106362 2025-04-16
3        0.963008 -0.107771 2025-04-15
4        0.981398 -0.091769 2025-04-16
5        0.978179 -0.094607 2025-04-08
6        1.040506 -0.102963 2025-04-08
7        1.034487 -0.083802 2025-04-08
8        1.026540 -0.081326 2025-04-08
9        1.030812 -0.073979 2025-04-08
10       1.024100 -0.066628 2025-04-08
11       0.969666 -0.084032 2025-04-08
12       0.977104 -0.082190 2025-04-08
13       1.016454 -0.040826 2025-04-16
14       1.029346 -0.051336 2025-04-08
15       0.983003 -0.103280 2025-04-08
16       1.074783 -0.042645 2025-03-28
17       0.995287 -0.072056 2025-04-16
18       0.965668 -0.099670 2025-04-16
19       1.061455 -0.079379 2025-04

In [18]:
import plotly.io as pio
pio.renderers.default = "browser"   
from batch.modeling.visualize import plot_driver_diagnostic_report

df_bt = _cal_driver_signal(df_oof_ev,df_oof_all,df_daily,START_DATE,END_DATE)
df_bt = df_bt[["sp500_ret", "macro_danger", "parachute_active", "risk_score", "1:Credit", "2:Bond", "3:Mix"]]

plot_driver_diagnostic_report(df_bt, start_date=START_DATE, end_date=END_DATE, lag=10)

In [19]:
#START_DATE = "2018-01-01"
#END_DATE = "2018-05-01"
pd.set_option("display.max.rows",None)
print(df_bt.loc[START_DATE:END_DATE])

            sp500_ret  macro_danger  parachute_active  risk_score  1:Credit  \
2025-03-03        NaN         False             False    0.369595  0.282573   
2025-03-04  -0.012235         False             False    0.281146  0.164301   
2025-03-05   0.011159         False             False    0.480667  0.374423   
2025-03-06  -0.017819         False             False    0.284745  0.173350   
2025-03-07   0.005521         False             False    0.484223  0.400407   
2025-03-10        NaN         False             False    0.400390  0.244752   
2025-03-11  -0.007568          True             False    0.738537  0.689998   
2025-03-12   0.004887          True             False    0.798277  0.756009   
2025-03-13  -0.013891         False             False    0.696856  0.650654   
2025-03-14   0.021266          True             False    0.814130  0.765777   
2025-03-17        NaN          True             False    0.742808  0.710373   
2025-03-18  -0.010654         False             Fals

### OLD

In [20]:
df = pd.concat([
    df_oof_all, 
    df_daily["^GSPC"].pct_change().ffill().rename("sp500_ret"),
    df_oof_ev["ev_rank"]
    ],sort=True,axis=1)
df = df.loc["2014-01-01":"2026-01-01"]
# シグナルの純粋な切り出し（閾値はモデルの出力に合わせて調整）
df['bond_active'] = (df['2:Bond'] > 0.4) # Bondの確信度が高い
df['credit_active'] = (df['1:Credit'] > 0.4) # Creditの確信度が高い

# 4つのセグメントに分類
df['segment'] = 'None'
df.loc[df['bond_active'] & ~df['credit_active'], 'segment'] = 'Bond_Only'
df.loc[~df['bond_active'] & df['credit_active'], 'segment'] = 'Credit_Only'
df.loc[df['bond_active'] & df['credit_active'], 'segment'] = 'Both'
df.loc[~df['bond_active'] & ~df['credit_active'], 'segment'] = 'Safe_Neutral'

# ev_rankとのクロス集計で期待値を算出
report = df.groupby(['ev_rank', 'segment'])['sp500_ret'].describe()
print(report)

                        count      mean       std       min       25%  \
ev_rank   segment                                                       
Safe      Bond_Only       1.0 -0.008428       NaN -0.008428 -0.008428   
          Safe_Neutral  941.0  0.001275  0.008494 -0.035920 -0.002417   
Neutral   Bond_Only     197.0 -0.000600  0.010104 -0.033688 -0.006362   
          Credit_Only    92.0  0.000637  0.009001 -0.037536 -0.003493   
          Safe_Neutral  686.0  0.000251  0.009176 -0.040395 -0.004322   
Caution   Bond_Only     292.0  0.000535  0.008718 -0.033688 -0.003253   
          Credit_Only   245.0  0.001787  0.014139 -0.043360 -0.003895   
          Safe_Neutral  134.0  0.000920  0.014129 -0.095113 -0.004156   
High Risk Bond_Only     172.0 -0.000016  0.013487 -0.048868 -0.003754   
          Both            4.0  0.009797  0.008385  0.005506  0.005506   
          Credit_Only   169.0 -0.002711  0.017378 -0.059750 -0.009412   
          Safe_Neutral   11.0  0.002880  0.005698 -

In [21]:
def _da_miss_credit_mix(df_oof_all,df_shap,df_driver):
    periods = [
        ("2013-05-24","2013-09-17"),("2013-10-02","2013-10-17"),
        ("2013-12-13","2013-12-18"),("2014-01-24","2014-02-10"),
        ("2015-03-04","2015-03-18"),("2017-01-04","2017-01-11"),
        ("2017-01-30","2017-02-13"),("2017-09-27","2017-10-02"),
    ]
    for start, end in periods:
        label="1:Credit"
        shap = df_shap[label].loc[start:end]
        top10_shap = shap.mean().sort_values(ascending=False).head(5)
        VIX_z252 = df_driver.loc[start:end, "VIX_z252"].mean()

        print(f"\nラベル{label}の{start}～{end}の平均確率と平均寄与度トップ5、およびVIX_z252の平均値")
        print(df_oof_all.loc[start:end].mean().round(2))
        print(f"\n{top10_shap}")
        print(f"\nVIX_z252の平均値: {VIX_z252}")

def _apply_probability_refinement(df_prob, df_shap, df_features):

    refined_prob = df_prob.copy()

    # 修正関数のリスト（今後、新しい修正モードが増えたらここに追加）
    refiners = [
        _refine_mode_interest_afterglow, # モードA: 金利の残像
        #_refine_mode_bond_vol_noise      # モードB: 債券ボラの過剰反応
    ]

    for refiner in refiners:
        refined_prob = refiner(refined_prob, df_shap, df_features)

    return refined_prob

def _refine_mode_interest_afterglow(df_prob, df_shap, df_features):

    VIX_SAFE_LIMIT = -0.5
    TP_SHAP_THRESHOLD = 0.35

    shap = df_shap["1:Credit"]

    # 1. 必要な中間変数を一括計算
    total_shap = shap.abs().sum(axis=1)
    # ゼロ除算を避けるために一応対策
    tp_ratio = shap['Term_Premium_z252'] / total_shap.replace(0, np.inf)
    vix_z = df_features['VIX_z252']

    # 2. 条件に合致するインデックスを特定する「マスク」を作成
    mask = (
        (df_prob["1:Credit"] >= 0.3) &
        (tp_ratio > TP_SHAP_THRESHOLD) &
        (vix_z < VIX_SAFE_LIMIT)
    )

    # 3. マスクされた行に対してのみ、ペナルティと移動量を一括計算
    if mask.any():
        # ペナルティ係数を計算 (0.0〜1.0の範囲にクリップ)
        penalty = ((vix_z[mask] - VIX_SAFE_LIMIT).abs() * tp_ratio[mask] * 2.0).clip(upper=1.0)

        # 移動量（デルタ）を算出
        delta = df_prob.loc[mask, '1:Credit'] * penalty

        # 確率の再分配
        df_prob.loc[mask, '1:Credit'] -= delta
        df_prob.loc[mask, '3:Mix'] += delta

    print(f"Mode A (Interest Afterglow) applied to: {mask.sum()} samples")

    return df_prob

def _refine_mode_bond_vol_noise(df_prob, df_shap, df_features):

    VIX_NEUTRAL_LIMIT = 0.5
    VOV_SHAP_THRESHOLD = 0.15
    
    shap = df_shap["1:Credit"]

    # 1. 必要な中間変数を一括計算
    total_shap = shap.abs().sum(axis=1)
    # MOVE_vovが判定に与えた影響のシェアを算出
    vov_ratio = shap['MOVE_vov'].abs() / total_shap.replace(0, np.inf)
    vix_z = df_features['VIX_z252']

    # 2. 条件に合致する「マスク」を作成
    # ・Credit確率が一定以上
    # ・債券ボラの寄与が閾値超え
    # ・VIXがパニック圏外（0.5以下）
    mask = (
        (df_prob['1:Credit'] >= 0.3) & 
        (vov_ratio > VOV_SHAP_THRESHOLD) & 
        (vix_z < VIX_NEUTRAL_LIMIT)
    )

    # 3. マスクされた行に対してのみ、一括で確率を移送
    if mask.any():
        # ペナルティ係数を計算 (Mode Bは最大0.8程度に抑える設定)
        penalty = (vov_ratio[mask] * 1.5).clip(upper=0.8)
        
        # 移動量（デルタ）を算出
        delta = df_prob.loc[mask, '1:Credit'] * penalty

        # 確率の再分配（Mode Bは債券要因なので 2:Bond へ移送）
        df_prob.loc[mask, '1:Credit'] -= delta
        df_prob.loc[mask, '2:Bond'] += delta

    # デバッグ用に修正件数を出力
    print(f"Mode B (Bond Vol Noise) applied to: {mask.sum()} samples")

    return df_prob